### Summarizing Mani Mama Lecture

* The youtube transcripts were not really good and there were a lot of mistakes. That is why we had to download the video and use openai-whisper library to get it transcribed. Use the transcribe.py to take the MP4 files and output the transcript into a text file.

In [2]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
import textwrap
import chromadb

DB_DIR = "./chromadb"
# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

vector_store = Chroma(collection_name="mani_mama_collection", client=chroma_client)


def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="mani_mama_collection", search_type="similarity", search_kwargs={"k": 5})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(textwrap.fill(result["answer"], width=70))
    matching_docs = result["source_documents"]
    return matching_docs

In [19]:
query = "Why is shama very important before vedantic meditation?" 
matching_docs = query_after_getting_matched_documents(query)
print("Matching document IDs:")
for doc in matching_docs:
    print('----------------------')
    print(textwrap.fill(str(doc.id), width=70))
    print(textwrap.fill(str(doc.page_content), width=70))
    print('----------------------')

Shama (sense control) is very important before Vedantic meditation
because, despite our ability to regulate and limit sense perceptions
through dhamma (self-control), some stimuli inevitably enter the mind
due to unavoidable external influences such as advertisements,
business activities, and societal pressures. Without controlling these
senses, mental turbulence persists, preventing a calm and undisturbed
state necessary for effective meditation and understanding of Vedanta.
Only when both shama and dhamma are effectively practiced can the mind
achieve steadiness (nidhityasanam), allowing for deeper concentration
and realization in Vedantic teachings.
Matching document IDs:
----------------------
videos/006.txt[0:40:00 - 0:41:00]
This is theep mantra. You climb upon this stone and like that stone,
you also be very firm. Now, now on I should strengthen my mind so hard
like this stone, which is not going to be affected whether it is sun,
moon or rain, whatever it is, I have to be firm i

In [17]:
query = "What makes one an adhikari for self knowledge?" 
matching_docs = query_after_getting_matched_documents(query)
print("Matching document IDs:")
for doc in matching_docs:
    print('----------------------')
    print(textwrap.fill(str(doc.id), width=70))
    print('----------------------')


According to the text, being called an **Adhikari** (the eligible
person) for self-knowledge requires possessing certain qualities known
as the *Sadhana Chaturthaka Sampatti*. These include:  1. **Viveka** –
discriminative ability or discernment, which allows one to distinguish
between what is truly the Self (*Atma*) and that which is not
(*Anatma*). 2. **Vairagya** – dispassion or detachment from worldly
desires and attachments. 3. **Shatka Sampati** (part of the four
qualifications) includes *Mumukshutvam* – a sincere craving for
liberation (*Moksha*) through self-knowledge.  Thus, an Adhikari must
exhibit these qualities: intellectual discernment, emotional
detachment, and a deep desire to attain Moksha. The text suggests that
without acquiring these characteristics—through practices such as
meditation, study of the Vedas (like Bhagavad Gita), and cultivation
of Vairagya—one cannot be eligible for self-knowledge or the deeper
teachings of Vedanta. Essentially, becoming an Adhikari i

In [21]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.documents import Document

ollama_model_name="granite4.1:3b"
llm = ChatOllama(model=ollama_model_name, base_url=None)

collection = chroma_client.get_or_create_collection(name="mani_mama_collection")

# Call the vector database for matching docs
results = collection.get(
    where={'chapter': {"$eq": "Chapter-02"}}
)

docs = []

m = results["documents"]
print("Got " + str(len(m)) + " matching documents")
for x in m:
    docs.append(Document(m[0]))
    

prompt_template = ChatPromptTemplate.from_messages(
    [
        ( 
            "You are a helpful assistant. Use the following pieces of retrieved context to generate a 1000 word summary \n\n"
            "{context}"
        )
    ]
)

chain = create_stuff_documents_chain(llm, prompt_template)

ans = chain.invoke({'context': docs})
print(textwrap.fill(ans, width=80))

Got 574 matching documents
The Sanskrit verse you provided, “Sri bhagavanuvachah asocyananvasocatvam
pragya‑vadamstha‑bhashase gata‑sūnagata‑sūmschah nānuśocanti‑panditah,” appears
to be a condensed or abbreviated expression related to the teachings of
Sri Vishnu (or another deity associated with “Sri bhagavan”). Without additional
context, it’s difficult to translate it word‑for‑word into English. However,
based on common usage in Vedic and devotional literature, the phrase seems to
convey a sentiment about:  1. **Asocya** – not being envious or hostile
(asocyanavat). 2. **Ananvasoca** – refraining from further agitation or strife.
3. **Pragya‑vatamstha‑bhasha** – speaking wisely and clearly (pragya = wisdom,
bhasa = speech). 4. **Gata‑sūnagata‑suṁschah** – moving forward, separating
oneself from others or obstacles (“gata” = gone/forward, “sūna” likely meaning
“child” or “offspring,” “suṁschah” may imply separation). 5.
**Nānuśocanti‑panditah** – those who are not envious (nānuśocant